# IMPORTACIONES

In [1]:
import pandas as pd
from sqlalchemy import create_engine, inspect
import pymysql
import os

# CARGA DE DATOS

Configurar los datos de conexión a la base de datos.

In [2]:
usuario = 'root'  
contrasena = os.environ.get('MYSQL_PASSWORD', '')  # define esta variable de entorno con tu contrasena
host = 'localhost'  
puerto = '3306'  
base_datos = 'dm_cardiologia'  

Crear la conexión.

In [3]:
# Crear la conexión usando SQLAlchemy
# Crear la conexion usando SQLAlchemy (la contrasena se lee de la variable de entorno MYSQL_PASSWORD)
con = create_engine(f"mysql+pymysql://{usuario}:{contrasena}@{host}:{puerto}/{base_datos}")

Si no supieramos qué tablas contiene la base de datos podríamos verlo así:

In [4]:
# Crear un objeto Inspector
inspector = inspect(con)

# Obtener y mostrar los nombres de las tablas
tablas = inspector.get_table_names()
print("Tablas en la base de datos:")
for tabla in tablas:
    print(tabla)

Tablas en la base de datos:
admisiones
diagnosticos
factores_riesgo
pacientes
pruebas_laboratorio


Cargamos cada una de las tablas en un dataframe.

In [5]:
admisiones = pd.read_sql("SELECT * FROM admisiones", con.connect())

diagnosticos = pd.read_sql("SELECT * FROM diagnosticos", con.connect())

factores_riesgo = pd.read_sql("SELECT * FROM factores_riesgo", con.connect())

pacientes = pd.read_sql("SELECT * FROM pacientes", con.connect())

pruebas_laboratorio = pd.read_sql("SELECT * FROM pruebas_laboratorio", con.connect())

Comprobamos que se han cargado bien los datos.

In [6]:
print(admisiones.shape)
print(diagnosticos.shape)
print(factores_riesgo.shape)
print(pacientes.shape)
print(pruebas_laboratorio.shape)

(15038, 8)
(15038, 29)
(15038, 9)
(15038, 5)
(15038, 10)


Cerramos todas las conexiones.

In [7]:
con.dispose()

# INTEGRACION DE DATOS

Vamos a ir integrando las tablas en base a lo que sabemos del modelo relacional.

In [8]:
# Paso 1: Unir `pacientes` y `admisiones` (FK: id_admision)
pacientes_admisiones = pacientes.merge(
    admisiones.drop(columns=['id_paciente']),
    on="id_admision",
    how="left"
)

In [9]:
print(pacientes_admisiones.shape)

(15038, 11)


In [10]:
# Paso 2: Unir el resultado con `factores_riesgo` (FK: id_admision, id_paciente)
pacientes_admisiones_factores = pacientes_admisiones.merge(
    factores_riesgo,
    on=["id_admision", "id_paciente"],
    how="left"
)

In [11]:
print(pacientes_admisiones_factores.shape)

(15038, 18)


In [12]:
# Paso 3: Unir el resultado con `pruebas_laboratorio` (FK: id_admision)
pacientes_admisiones_factores_pruebas = pacientes_admisiones_factores.merge(
    pruebas_laboratorio,
    on="id_admision",
    how="left"
)

In [13]:
print(pacientes_admisiones_factores_pruebas.shape)

(15038, 27)


In [14]:
# Paso 4: Unir el resultado con `diagnosticos` (FK: id_admision)
df = pacientes_admisiones_factores_pruebas.merge(
    diagnosticos,
    on="id_admision",
    how="left"
)

In [15]:
print(df.shape)

(15038, 55)


Creamos un caché (buena práctica si los datos fueran muy voluminosos)

In [19]:
import pickle
df.to_pickle('./data/df.pickle')